# 🧠 Building Advanced Transformers — Self-Attention for Time Series

In this notebook I implement a Transformer **encoder from scratch** in Keras and use it to forecast a univariate time series. I build every piece myself — the multi-head self-attention, the Transformer block, the encoder stack — then assemble them into a full model, train it, and run experiments.

## 📋 Overview

Transformers replaced recurrence (RNN/LSTM) with **attention**: instead of passing a hidden state step by step, every position looks at every other position in one shot. That makes them parallelisable and great at capturing long-range structure.

🎯 **What I do here:**

| Step | What I build |
|---|---|
| 📥 Part 1 | Generate a synthetic price series and window it into supervised samples |
| 🏗️ Part 2 | Implement multi-head self-attention, a Transformer block, and the encoder stack from scratch |
| ⚙️ Part 3 | Assemble the full model, train it, and visualise predictions |
| 🧪 Part 4 | Experiments: dropout, batch size, and activation function |

**📡 Engineering analogy.** Self-attention is essentially an *adaptive beamformer*. A phased array weights each antenna element to steer a beam toward the direction carrying signal; self-attention learns weights over each position in the sequence to "steer" the model toward the time steps that matter for the current prediction. The attention matrix is the steering vector — except here the weights are learned and content-dependent, not fixed by geometry.

## 🧩 Theory

### 🔢 Scaled dot-product attention

Each input vector is projected into three roles — **query** $Q$, **key** $K$, and **value** $V$. The query of a position is compared against the keys of all positions (a dot product = similarity), the scores are scaled and softmaxed into weights, and those weights mix the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

Where:
- $Q \in \mathbb{R}^{n \times d_k}$ = queries (what each position is looking for)
- $K \in \mathbb{R}^{n \times d_k}$ = keys (what each position offers)
- $V \in \mathbb{R}^{n \times d_v}$ = values (the content actually mixed)
- $d_k$ = key dimension; dividing by $\sqrt{d_k}$ keeps the dot products from exploding so the softmax stays in a sensitive range

📡 The $QK^\top$ term is a **correlation matrix** between positions — exactly like cross-correlating two signals to see how much they overlap.

### ➕ Multi-head attention

One attention map can only emphasise one kind of relationship. So I run $h$ attention operations in parallel ("heads"), each on a $d/h$-dimensional slice, then concatenate:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^{O}, \qquad \text{head}_i = \text{Attention}(Q W_i^{Q}, K W_i^{K}, V W_i^{V})$$

📡 Like a **multi-band receiver**: each head tunes to a different "frequency" of relationship in the sequence, and the output projection $W^O$ combines them.

### 🔄 Residual connections + layer normalisation

Each sub-layer is wrapped as $\text{LayerNorm}(x + \text{Sublayer}(x))$. The residual $x +$ lets gradients flow straight through deep stacks; layer norm standardises each vector:

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

📡 The residual path is a **bypass / feed-through line**, and layer norm is **automatic gain control** keeping every stage at a stable operating point.

## Part 1 — 📥 Data: Synthetic Price Series & Windowing

I first install the libraries, then generate a synthetic "stock price" signal — a linear trend plus Gaussian noise — and slice it into overlapping windows so the Transformer can learn to predict the next step from the previous 100.

In [ ]:
%pip install tensorflow pyarrow
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install requests

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import requests
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout

### 🏗️ Generate the synthetic signal

I build a deterministic trend with `np.linspace` (100 → 200) and add zero-mean noise. Fixing the seed makes the experiment reproducible. 📡 This is a classic **trend + additive white Gaussian noise (AWGN)** model — the same decomposition I'd use to describe a slowly drifting carrier with thermal noise on top.

In [ ]:
import numpy as np
import pandas as pd

# Create a synthetic stock price dataset
np.random.seed(42)
data_length = 2000  # Adjust data length as needed
trend = np.linspace(100, 200, data_length)
noise = np.random.normal(0, 2, data_length)
synthetic_data = trend + noise

# Create a DataFrame and save as 'stock_prices.csv'
data = pd.DataFrame(synthetic_data, columns=['Close'])
data.to_csv('stock_prices.csv', index=False)
print("Synthetic stock_prices.csv created and loaded.")

### 📥 Normalise and window the data

Two preprocessing steps:

- **`MinMaxScaler`** rescales values into $[0, 1]$ via $x' = \dfrac{x - x_{\min}}{x_{\max} - x_{\min}}$. Neural nets train far better on bounded inputs — 📡 think of it as **normalising signal power before the ADC** so nothing clips and every feature sits in the same dynamic range.
- **`create_dataset`** turns the 1-D series into supervised pairs using a sliding window of `time_step = 100`: each $X$ is 100 consecutive points and each $Y$ is the very next point. This is a **tapped delay line** — the model sees the last 100 samples and predicts sample 101.

In [ ]:
# Load the dataset
data = pd.read_csv('stock_prices.csv')
data = data[['Close']].values

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data)

# Prepare the data for training
def create_dataset(data, time_step=1):
    X, Y = [], []

    for i in range(len(data)-time_step-1):
        a = data[i:(i+time_step), 0]
        X.append(a)
        Y.append(data[i + time_step, 0])
    return np.array(X), np.array(Y)

time_step = 100
X, Y = create_dataset(data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape) 

After windowing, `X` has shape `(samples, 100, 1)` — a batch of length-100 sequences with one feature each — and `Y` holds the next-step target for every window.

## Part 2 — 🏗️ Building the Transformer

Now I build the architecture bottom-up: attention → block → encoder stack.

### 🔢 Multi-head self-attention

I implement the attention math from scratch as a custom Keras `Layer`. The three dense projections create $Q$, $K$, $V$; `split_heads` reshapes them into `num_heads` parallel heads; `attention` computes the scaled-dot-product formula; then the heads are merged and projected back with `combine_heads`.

In [ ]:
class MultiHeadSelfAttention(Layer):

    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        self.combine_heads = Dense(embed_dim)


    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)
        attention, _ = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

Reading my own code back:

- `attention(...)` is the literal implementation of $\text{softmax}(QK^\top/\sqrt{d_k})\,V$ — `transpose_b=True` does the $K^\top$, and the division is the $\sqrt{d_k}$ scaling.
- `split_heads(...)` reshapes `(batch, seq, embed_dim)` into `(batch, heads, seq, projection_dim)` so all heads attend in parallel.
- `call(...)` projects the input to Q/K/V, splits into heads, attends, then concatenates and mixes the heads back to `embed_dim`.

### 🔄 Transformer block

The block wraps attention and a position-wise feed-forward network, each inside a residual + layer-norm shell. Dropout regularises both sub-layers. This is the "Add & Norm" pattern from the theory section made concrete.

In [ ]:
class TransformerBlock(Layer):

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)


    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

The flow is exactly $\text{out}_1 = \text{LayerNorm}(x + \text{Attn}(x))$ followed by $\text{out}_2 = \text{LayerNorm}(\text{out}_1 + \text{FFN}(\text{out}_1))$. The feed-forward net expands to `ff_dim` with ReLU then projects back — 📡 a per-position **non-linear filter stage** applied identically to every time step.

### 🏗️ Encoder layer

An encoder layer is structurally identical to the Transformer block — I define it separately because it's the reusable unit the encoder stacks. Same Add & Norm pattern, same two sub-layers.

In [ ]:
class EncoderLayer(Layer):

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(EncoderLayer, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)



    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

### 🔢 Transformer encoder (full stack)

The encoder chains `num_layers` Transformer blocks. Each layer refines the representation produced by the previous one — 📡 like a **cascade of filter stages**, where depth lets the model compose increasingly abstract relationships across the sequence. I re-declare the building blocks in this cell so it runs standalone, then verify the output shape with a random input.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Dropout

class MultiHeadSelfAttention(Layer):
    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        self.combine_heads = Dense(embed_dim)


    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights


    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])


    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)
        attention, _ = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)


    def call(self, inputs, training):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerEncoder(Layer):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerEncoder, self).__init__()
        self.num_layers = num_layers
        self.embed_dim = embed_dim
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)]
        self.dropout = Dropout(rate)

    def call(self, inputs, training=False):
        x = inputs
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training=training)
        return x

# Example usage
embed_dim = 128
num_heads = 8
ff_dim = 512
num_layers = 4

transformer_encoder = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim)
inputs = tf.random.uniform((1, 100, embed_dim))
outputs = transformer_encoder(inputs, training=False)  # Use keyword argument for 'training'
print(outputs.shape)  # Should print (1, 100, 128)

The output shape `(1, 100, 128)` confirms the encoder preserves the sequence length and embedding dimension — it transforms the representation in place rather than collapsing it, so I can stack as many layers as I like.

## Part 3 — ⚙️ Assemble, Train & Evaluate

### 🏗️ Build and compile the model

The raw input is `(100, 1)` — one feature per step. A `Dense(embed_dim)` projection lifts each step into a 128-dim embedding (the Transformer's working space), the encoder processes it, `Flatten` unrolls the `(100, 128)` output into one long vector, and a final `Dense(1)` produces the next-step prediction. I compile with **Adam** and **MSE** loss, the natural choice for regression:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
# Define the necessary parameters

embed_dim = 128
num_heads = 8
ff_dim = 512
num_layers = 4

# Define the Transformer Encoder
transformer_encoder = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim)

# Build the model
input_shape = (X.shape[1], X.shape[2])
inputs = tf.keras.Input(shape=input_shape)

# Project the inputs to the embed_dim
x = tf.keras.layers.Dense(embed_dim)(inputs)
encoder_outputs = transformer_encoder(x)
flatten = tf.keras.layers.Flatten()(encoder_outputs)
outputs = tf.keras.layers.Dense(1)(flatten)
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Summary of the model
model.summary()

### 🎯 Train the model

I fit for 20 epochs with batch size 32. Each step Adam nudges the weights to shrink the MSE between predicted and actual next-step prices.

In [ ]:
# Train the model
model.fit(X, Y, epochs=20, batch_size=32)

### 📈 Evaluate and predict

I run the model over every window, then **inverse-transform** the predictions back to real price units (undoing the MinMax scaling) so the plot is interpretable. Predictions are offset by `time_step` because the first prediction corresponds to the 101st point.

In [ ]:
# Make predictions
predictions = model.predict(X)
predictions = scaler.inverse_transform(predictions)

# Prepare true values for comparison
true_values = scaler.inverse_transform(data.reshape(-1, 1))

# Plot the predictions vs true values
import matplotlib.pyplot as plt

plt.plot(true_values, label='True Data')
plt.plot(np.arange(time_step, time_step + len(predictions)), predictions, label='Predictions')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.title('Predictions vs True Data (Both Scaled Back)')
plt.show()

The prediction curve should track the underlying trend closely, with the model effectively averaging out the injected noise — 📡 behaving like a **learned smoothing filter** over the series.

## Part 4 — 🧪 Experiments

Three controlled experiments to see how design choices move performance. I keep everything else fixed and change one variable at a time.

### 🧪 Experiment 1 — Add dropout to fight overfitting

🎯 **Goal:** insert a `Dropout(0.5)` after the `Flatten` layer. Dropout randomly zeroes half the activations during training, forcing the network not to rely on any single feature — 📡 like **antenna diversity**, where the system stays robust because it never depends on one element. I rebuild the head with dropout, retrain, and evaluate.

In [ ]:
from tensorflow.keras.layers import Dropout

# Add a dropout layer after the Flatten layer
flatten = tf.keras.layers.Flatten()(encoder_outputs)
dropout = Dropout(0.5)(flatten)
outputs = tf.keras.layers.Dense(1)(dropout)

# Build the model
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Train the model
model.fit(X, Y, epochs=20, batch_size=32)

# Evaluate the model
loss = model.evaluate(X, Y)
print(f'Test loss: {loss}')

### 🧪 Experiment 2 — Compare batch sizes

🎯 **Goal:** train with batch size 16 and then 64, and compare. Smaller batches give noisier but more frequent gradient updates (often better generalisation, slower per epoch); larger batches give smoother, faster updates that may need more epochs to reach the same loss. 📡 It's the classic **integration-time trade-off**: average over fewer samples and you react fast but noisily; average over more and you're smooth but sluggish.

In [ ]:
# Train the model with batch size 16
model.fit(X, Y, epochs=20, batch_size=16)

# Evaluate the model
loss = model.evaluate(X, Y)
print(f'Test loss with batch size 16: {loss}')

# Train the model with batch size 64
model.fit(X, Y, epochs=20, batch_size=64)

# Evaluate the model
loss = model.evaluate(X, Y)
print(f'Test loss with batch size 64: {loss}')

### 🧪 Experiment 3 — Change the output activation

🎯 **Goal:** swap the final `Dense` activation to `tanh` and observe the effect. `tanh` squashes outputs into $(-1, 1)$:

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$$

Since my targets are MinMax-scaled into $[0, 1]$, a `tanh` head can only reach the upper half of its range — a useful reminder that the **output activation must match the target's range**. I rebuild with `tanh`, retrain, and check the loss.

In [ ]:
# Change the activation function of the Dense layer to tanh
outputs = tf.keras.layers.Dense(1, activation='tanh')(flatten)

# Build the model
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Train the model
model.fit(X, Y, epochs=20, batch_size=32)

# Evaluate the model
loss = model.evaluate(X, Y)
print(f'Test loss with tanh activation: {loss}')

## 📊 Summary

| 🧩 Component | Role | 📡 Engineering analogy |
|---|---|---|
| Input embedding (`Dense`) | Lifts each scalar step into a 128-dim vector | Mapping a sample onto a feature/basis set |
| Multi-head self-attention | Each position attends to all positions, in parallel heads | Adaptive beamformer / multi-band correlator |
| Scaling by $\sqrt{d_k}$ | Keeps dot products in the softmax's sensitive range | Gain staging before a non-linearity |
| Feed-forward network | Per-position non-linear transform | Per-tap non-linear filter stage |
| Residual + LayerNorm | Stable gradient flow, normalised activations | Bypass line + automatic gain control |
| Encoder stack (`num_layers`) | Refines representation layer by layer | Cascade of filter stages |
| Dropout | Regularisation against overfitting | Antenna / path diversity |
| Adam + MSE | Optimiser + regression loss | Gradient-based tuning toward min error |

✅ **What I built and learned:**
- Implemented scaled dot-product and multi-head self-attention **from scratch** in Keras.
- Composed them into a Transformer block, encoder layer, and full encoder stack.
- Applied a Transformer to **time-series forecasting** with proper scaling and windowing.
- Saw first-hand how dropout, batch size, and output activation shift performance.

| Hyperparameter | Value used |
|---|---|
| `embed_dim` | 128 |
| `num_heads` | 8 |
| `ff_dim` | 512 |
| `num_layers` | 4 |
| `time_step` (window) | 100 |
| Optimiser / loss | Adam / MSE |

## 🧪 Sandbox

Space to experiment further. Ideas worth trying:

- ➕ **Add positional encoding.** This implementation has none, so the encoder is permutation-invariant — it can't tell step 1 from step 99. Inject sinusoidal positions $PE_{(pos,2i)} = \sin\!\big(pos / 10000^{2i/d}\big)$ and see if forecasting improves. 📡 This is literally giving each sample a *timestamp / phase reference*.
- 🔄 **Sweep `num_layers` and `num_heads`** and plot loss vs depth — where do diminishing returns kick in?
- 📈 **Proper train/test split.** Right now I evaluate on training data; hold out the last 20% to measure real generalisation.
- 🎯 **Multi-step forecasting:** predict the next 5 points instead of 1.
- ⚙️ **Visualise the attention weights** (`weights` returned by `attention`) as a heatmap to see which past steps the model leans on.

In [ ]:
# 🧪 Sandbox — experiment freely here
